# 04. 여러 구조 비교하기

            03번에서 만든 후보 edge를 가지고 여러 graph 구조를 비교합니다.

            이 노트북의 핵심 질문은 이것입니다.

            > 같은 데이터라도 algorithm을 바꾸면 선택되는 network가 어떻게 달라질까?


## 오늘 사용할 말

- graph(그래프): 점과 선으로 이루어진 연결 구조
- node(꼭짓점): 지도 위의 역 후보 지점
- edge(변): 두 지점을 연결하는 하나의 경로
- weight(가중치): 어떤 edge가 좋은지 나쁜지 판단하는 점수
- normalization(정규화): 서로 단위가 다른 값을 비교 가능한 점수로 바꾸는 일
- MST, minimum spanning tree(최소신장수형도): 모든 node를 연결하되 총 비용을 작게 만드는 기본 구조
- shortest path(최단경로): graph 안에서 두 node 사이를 가장 짧게 가는 경로
- stretch(우회율): 선택한 구조에서 얼마나 돌아가는지 나타내는 값
- t-spanner(t-스패너): 너무 많이 돌아가지 않도록 edge를 추가하는 방법


## 1. 준비하기

03번 노트북에서 만든 `03_edge_scores.csv`와 `03_candidate_edges.csv`를 읽습니다. 학생이 CSV를 열 필요는 없습니다.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (path / "analysis/student_helpers.py").is_file():
        PROJECT_ROOT = path
        break
    if (path / "student_helpers.py").is_file():
        PROJECT_ROOT = path
        break

if (PROJECT_ROOT / "analysis").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT / "analysis"))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

from student_helpers import *

OUT = output_dir(PROJECT_ROOT)
print("작업 폴더:", PROJECT_ROOT)
print("결과 저장 폴더:", OUT)


In [ ]:
stations, _ = load_current_data(PROJECT_ROOT)
reference_edges = read_csv(OUT / "03_edge_scores.csv")
candidate = read_csv(OUT / "03_candidate_edges.csv")

for rows in [reference_edges, candidate]:
    for edge in rows:
        for key in ["pair_order", "from_order", "to_order"]:
            edge[key] = int(edge[key])
        for key in numeric_edge_columns():
            if key in edge:
                edge[key] = float(edge[key])

nodes = components(station_ids(stations), reference_edges)[0]
print("기준 후보 edge:", len(reference_edges))
print("03번 후보 network edge:", len(candidate))
print("분석할 node:", len(nodes))


## 2. 비교할 구조 만들기

여기서는 네 가지 구조를 비교합니다.

- `CURRENT_03_CANDIDATE`: 03번에서 만든 후보 network
- `MST`: 가장 얇게 모든 node를 연결하는 구조
- `T2_SPANNER`: 너무 돌아가지 않도록 edge를 더한 구조
- `DEGREE_LIMIT_3`: 한 node에 너무 많은 edge가 몰리지 않도록 제한한 구조


In [ ]:
structures = {
    "CANDIDATE_GRAPH": reference_edges,
    "CURRENT_03_CANDIDATE": candidate,
    "MST": mst(nodes, reference_edges, "scenario_cost"),
    "T2_SPANNER": greedy_spanner(nodes, reference_edges, 2.0),
    "DEGREE_LIMIT_3": degree_limited_kruskal(nodes, reference_edges, 3),
    "DEGREE_LIMIT_4": degree_limited_kruskal(nodes, reference_edges, 4),
}

for name, selected in structures.items():
    print(name, "edge 수:", len(selected))


## 3. 구조를 숫자로 비교하기

edge 수가 적으면 단순하지만, 돌아가는 길이 길어질 수 있습니다.

그래서 다음 값을 같이 봅니다.

- `edge_count`: 선택된 edge 수
- `total_w1_distance_km`: 선택된 edge의 총 거리
- `average_shortest_path_km`: network 안에서 평균적으로 이동해야 하는 거리
- `max_distance_stretch`: 가장 심하게 돌아가는 경우의 우회율


In [ ]:
metrics = []
stretches = []
edge_rows = []

for name, selected in structures.items():
    row, stretch_rows = structure_metrics(name, nodes, reference_edges, selected)
    metrics.append(row)
    stretches.extend(stretch_rows)
    for edge in selected:
        item = dict(edge)
        item["structure"] = name
        edge_rows.append(item)

write_csv(OUT / "04_structure_comparison.csv", metrics)
write_csv(OUT / "04_stretch_pairs.csv", stretches)
write_csv(OUT / "04_structure_edges.csv", edge_rows)

print_table(
    metrics,
    ["structure", "edge_count", "total_w1_distance_km", "average_shortest_path_km", "max_distance_stretch"],
    limit=10,
)


## 4. t-spanner에서 t 값을 바꿔보기

`t`가 작으면 “많이 돌아가지 말라”는 뜻이 강해져 edge가 더 많이 필요합니다.

아래 결과에서 `t`가 커질수록 edge 수가 어떻게 변하는지 확인해보세요.


In [ ]:
spanner_rows = []
for t in [1.25, 1.5, 2.0, 2.5, 3.0]:
    selected = greedy_spanner(nodes, reference_edges, t)
    row, _ = structure_metrics(f"T{t:g}_SPANNER", nodes, reference_edges, selected)
    row["t"] = t
    spanner_rows.append(row)

write_csv(OUT / "04_spanner_sensitivity.csv", spanner_rows)
write_json(OUT / "04_structure_summary.json", {"status": "PASS", "structures": metrics, "classification": "LECTURE_STRUCTURE_COMPARISON_NOT_FINAL_ROUTE"})
print_table(spanner_rows, ["t", "edge_count", "average_shortest_path_km", "max_distance_stretch"], limit=10)


## 5. 그림으로 비교하기

그래프는 숫자만 보면 감이 잘 오지 않습니다. 아래 셀은 구조별 핵심 지표를 막대그래프로 저장합니다.

생각해볼 질문:

- edge가 적은 구조가 항상 좋은 구조일까요?
- `max_distance_stretch`가 큰 구조는 어떤 문제가 있을까요?


In [ ]:
names = [m["structure"] for m in metrics if m["structure"] != "CANDIDATE_GRAPH"]
table = {m["structure"]: m for m in metrics}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
specs = [
    ("edge_count", "Edge count"),
    ("total_w1_distance_km", "Total W1 km"),
    ("average_shortest_path_km", "Average shortest km"),
    ("max_distance_stretch", "Max stretch"),
]
for ax, (field, title) in zip(axes.flat, specs):
    ax.bar(range(len(names)), [table[n][field] for n in names], color="#0f766e")
    ax.set_xticks(range(len(names)), [n.replace("_", "\n") for n in names], fontsize=7)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUT / "04_structure_comparison.png", dpi=180)
plt.show()
